# Post 1: the simpler models on Imagenette

The steps on the way to the ResNet: a linear model, an MLP and a small CNN. Each trains
with the ResNet's exact recipe, so the only thing that differs is the model.

Set the `RUNS_TO_TRAIN` environment variable to a comma-separated list of names to train
only some of them. The small CNN is slow on a laptop CPU, so it's meant for a Colab TPU:
see `COLAB.md`. As with the ResNet, checkpoints go to a GCS bucket and metrics to W&B, so a
lost runtime can be restarted and carry on.

## Setup

On Colab this clones the branch below and installs it. `BRANCH` must already be pushed,
because the runtime sees GitHub, not your working copy. Locally it does nothing.

The repo needs a JAX recent enough to have `jax.set_mesh`. If the runtime's is older,
this upgrades it before anything imports JAX.

In [ ]:
import os
import pathlib
import sys

BRANCH = "p01-data-pipeline"

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("/content/toylib").exists():
        !git clone --depth 1 --branch {BRANCH} https://github.com/anujkhare/toylib.git /content/toylib
    %cd /content/toylib
    !git pull --ff-only origin {BRANCH}
    !pip install -q -e ".[harness,viz,gcs]"
    has_set_mesh = !python -c "import jax; print(hasattr(jax, 'set_mesh'))"
    if has_set_mesh[-1] != "True":
        !pip install -q -U "jax[tpu]"

## Sign in to GCS and Weights & Biases

`colab exec` can't answer prompts, so this cell asks for nothing. `COLAB.md` shows how a
service-account key and the W&B key reach the runtime as files. The key isn't passed with `colab exec --env`, because that saves it in plain
text in the CLI's local history.

If the runtime can't read the bucket, this stops here rather than failing hours later.

In [ ]:
os.environ.setdefault("GOOGLE_CLOUD_PROJECT", "practicalml-db49f")  # For the bucket.

WANDB_PROJECT = "toylib-p01-imagenette"
WANDB_ENTITY = "anujkhare18"
BUCKET = "gs://tinystories-checkpoints"
CHECKPOINT_ROOT = f"{BUCKET}/toylib-p01-imagenette"
WANDB_KEY_FILE = pathlib.Path("/content/.wandb_key")

if IN_COLAB:
    project = os.environ.get("GOOGLE_CLOUD_PROJECT")
    if project:
        !gcloud config set project {project}
    bucket_access = !gcloud storage ls {BUCKET}/ 2>&1 | head -1
    if not bucket_access or not bucket_access[0].startswith("gs://"):
        raise RuntimeError(f"Can't read the bucket, so check the service-account key; see COLAB.md: {bucket_access}")
    if "WANDB_API_KEY" not in os.environ:
        if not WANDB_KEY_FILE.exists():
            raise RuntimeError(f"Upload the W&B key to {WANDB_KEY_FILE} first; see COLAB.md.")
        os.environ["WANDB_API_KEY"] = WANDB_KEY_FILE.read_text().strip()

## The runs

| Run | Model | Parameters |
| --- | --- | --- |
| `linear` | One linear layer on the pixels | 768,000 |
| `mlp` | One hidden layer of 256 units | 19,663,360 |
| `small_cnn` | Four convolutions with pooling, flatten, one linear layer | 368,842 |

All three use 200 epochs, flips, random resized crops, a 40x40 cutout, and AdamW at 1e-3
with cosine decay and weight decay 0.1. The small CNN has no skip connections and no
normalization, and its flatten head reads features from fixed positions.

The harness drops a partial batch, so evaluation during training covers 3,600 of the
3,925 test images, picked at random. The final accuracy below uses all of them, plus the
clean training images, to show how much each model memorizes.

In [ ]:
import jax
import numpy as np
import optax
from absl import logging as absl_logging

from toylib.harness import inference
from toylib.harness.experiment import (
    CheckpointConfig,
    EvalConfig,
    Experiment,
    MultiOptimizerConfig,
    OptimizerConfig,
    Task,
    TrainingConfig,
    WandBLoggerConfig,
)
from toylib.harness.metrics import Loss
from toylib_projects.p01_cnn_mnist import data, metrics, model

IMAGE_SHAPE = (160, 160, 3)
RUNS = {
    "linear": (model.build_mlp, model.MLPConfig(IMAGE_SHAPE)),
    "mlp": (model.build_mlp, model.MLPConfig(IMAGE_SHAPE, hidden_dims=(256,))),
    "small_cnn": (
        model.build_model,
        model.ModelConfig(IMAGE_SHAPE, channels=(32, 64, 128, 128), pools=(True,) * 4),
    ),
}
if "RUNS_TO_TRAIN" in os.environ:
    RUNS = {name: RUNS[name] for name in os.environ["RUNS_TO_TRAIN"].split(",")}
EPOCHS = 200
BATCH_SIZE = 128
EVAL_BATCH_SIZE = 400
CUTOUT_SIZE = 40
WEIGHT_DECAY = 0.1
CHECKPOINT_EVERY_EPOCHS = 50  # The MLP's checkpoints are over 200 MB each.

absl_logging.set_verbosity(absl_logging.ERROR)  # quiet orbax's read-only notices
print("JAX", jax.__version__, "| devices:", jax.devices())
if IN_COLAB and jax.default_backend() == "cpu":
    raise RuntimeError("No accelerator on this runtime, so training would take days.")

In [ ]:
def make_experiment(name):
    """One run through the harness, logged to W&B with checkpoints in GCS."""
    train_ds, test_ds = data.make_imagenette_datasets(
        batch_size=BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE, cutout_size=CUTOUT_SIZE
    )
    steps_per_epoch = train_ds.batches_per_epoch
    total_steps = EPOCHS * steps_per_epoch
    run_id = f"{name}_e{EPOCHS}"
    out_dir = pathlib.Path(model.__file__).parent / "outputs" / f"imagenette_{run_id}"

    learning_rate = optax.cosine_decay_schedule(1e-3, total_steps)
    optimizer_config = MultiOptimizerConfig(
        [OptimizerConfig("adamw", optax.adamw(learning_rate, weight_decay=WEIGHT_DECAY))],
        optimizer_for_param=lambda path: "adamw",
    )
    task_metrics = [Loss(), metrics.Accuracy()]
    return Experiment(
        train_task=Task("train", train_ds, metrics=task_metrics),
        eval_task=Task("test", test_ds, metrics=task_metrics),
        model_factory=RUNS[name][0],
        forward_fn=model.forward,
        model_config=RUNS[name][1],
        # +1: the harness evaluates and saves a step before counting it, so this
        # makes the last epoch's evaluation and checkpoint happen.
        training_config=TrainingConfig(
            optimizer_config=optimizer_config, max_steps=total_steps + 1
        ),
        eval_config=EvalConfig(
            eval_interval_steps=steps_per_epoch, num_eval_steps=test_ds.batches_per_epoch
        ),
        checkpoint_config=CheckpointConfig(
            save_interval_steps=CHECKPOINT_EVERY_EPOCHS * steps_per_epoch,
            max_to_keep=2,
            checkpoint_dir=f"{CHECKPOINT_ROOT}/{run_id}",
            checkpoint_dataset_iterator=True,  # Resume mid-epoch with the same order.
        ),
        logger_config=WandBLoggerConfig(
            project_name=WANDB_PROJECT,
            user_name=WANDB_ENTITY,
            log_dir=str(out_dir),
            run_id=run_id,  # A repeat of this run resumes the same W&B run.
            train_log_interval_steps=50,
            also_log_to_file=True,
        ),
    )

## Train

Prints a W&B link for each run. Each one starts from its newest checkpoint in the bucket if
it has one, and a finished run only repeats the final measurements.

In [ ]:
def accuracy(net, arrays):
    """Accuracy over a whole split, on clean centre crops."""
    info = data.DATASETS["imagenette"]
    images = data.preprocess(data.center_crop(arrays["image"], IMAGE_SHAPE[0]), info)
    logits = inference.batched_apply(
        lambda m, b: m(b["image"]), net, {"image": images}, EVAL_BATCH_SIZE
    )
    return float((logits.argmax(-1) == arrays["label"]).mean())


results = {}
for name in RUNS:
    exp = make_experiment(name)
    try:
        exp.init_state()
        resumed = exp.resume()
        print(f"\n{name}: " + (f"resuming at step {exp.step}" if resumed else "starting fresh")
              + f" of {exp.training_config.max_steps}")
        exp.outer_loop()
        results[name] = (
            accuracy(exp.model, exp.train_task.dataset.arrays),
            accuracy(exp.model, exp.eval_task.dataset.arrays),
        )
    finally:
        exp.cleanup()  # Waits for W&B to finish syncing.
    print(f"{name}: clean train accuracy {results[name][0]:.4f}, test accuracy {results[name][1]:.4f}")

## Results

The ResNet with the same recipe reached 100% on clean training images and 87.4% on test.
Checkpoints stay in the bucket under `CHECKPOINT_ROOT`, in one folder per run.

In [ ]:
for name, (train_accuracy, test_accuracy) in results.items():
    print(f"{name:10s} clean train {train_accuracy:.4f}  test {test_accuracy:.4f}")